In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 1995
month = 5


## Temperature and Salinity download 
* extrapolate temperature into the undefined boxes
* example code:

temp = xr.open_dataset(…).temp  
invalid_mask = …  
temp_extrap = xr.where(~invalid_mask, temp, temp.rolling(lon=3, lat=3, z=3, center=True, min_periods=1).mean())  

In [3]:
import copernicusmarine
import xarray as xr
import matplotlib.pyplot as plt
from cmocean import cm 
import numpy as np
import pandas as pd

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Call CMEMS data

In [4]:
from datetime import datetime
import calendar

In [5]:
last_day = calendar.monthrange(year, month)[1]
start_date = f"{year}-{month:02d}-01T00:00:00"
end_date = f"{year}-{month:02d}-{last_day:02d}T23:59:59"

In [6]:
data_request = {
   "dataset_id_plume" : "cmems_mod_glo_phy_my_0.083deg_P1D-m",
   "dataset_version": "202311",
   "longitude" : [-100, -0], 
   "latitude" : [-50, 50],
   "time" : [start_date, end_date],
   "variables" : ["so","thetao"]
}

# Load xarray dataset
ds = copernicusmarine.open_dataset(
    dataset_id = data_request["dataset_id_plume"],
    minimum_longitude = data_request["longitude"][0],
    maximum_longitude = data_request["longitude"][1],
    minimum_latitude = data_request["latitude"][0],
    maximum_latitude = data_request["latitude"][1],
    start_datetime = data_request["time"][0],
    end_datetime = data_request["time"][1],
    variables = data_request["variables"],
    username = 'alizarbe',
    password = 'DoNuT_120197',
    chunk_size_limit = -1
)

# Print loaded dataset information
ds

INFO - 2025-09-18T10:41:08Z - Selected dataset version: "202311"


INFO - 2025-09-18T10:41:08Z - Selected dataset part: "default"


<xarray.Dataset> Size: 36GB
Dimensions:    (depth: 50, latitude: 1201, longitude: 1201, time: 31)
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 5kB -50.0 -49.92 -49.83 ... 49.83 49.92 50.0
  * longitude  (longitude) float32 5kB -100.0 -99.92 -99.83 ... -0.08333 0.0
  * time       (time) datetime64[ns] 248B 1995-05-01 1995-05-02 ... 1995-05-31
Data variables:
    so         (time, depth, latitude, longitude) float64 18GB dask.array<chunksize=(2, 50, 512, 1201), meta=np.ndarray>
    thetao     (time, depth, latitude, longitude) float64 18GB dask.array<chunksize=(2, 50, 512, 1201), meta=np.ndarray>
Attributes:
    history:      2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    references:   http://www.mercator-ocean.fr
    source:       MERCATOR GLORYS12V1
    institution:  MERCATOR OCEAN
    Conventions:  CF-1.4
    title:        daily mean fields from Global Ocean Physics Analysis and Fo...
    comment:      CMEMS product

In [7]:
print(ds)

<xarray.Dataset> Size: 36GB
Dimensions:    (depth: 50, latitude: 1201, longitude: 1201, time: 31)
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 5kB -50.0 -49.92 -49.83 ... 49.83 49.92 50.0
  * longitude  (longitude) float32 5kB -100.0 -99.92 -99.83 ... -0.08333 0.0
  * time       (time) datetime64[ns] 248B 1995-05-01 1995-05-02 ... 1995-05-31
Data variables:
    so         (time, depth, latitude, longitude) float64 18GB dask.array<chunksize=(2, 50, 512, 1201), meta=np.ndarray>
    thetao     (time, depth, latitude, longitude) float64 18GB dask.array<chunksize=(2, 50, 512, 1201), meta=np.ndarray>
Attributes:
    history:      2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    references:   http://www.mercator-ocean.fr
    source:       MERCATOR GLORYS12V1
    institution:  MERCATOR OCEAN
    Conventions:  CF-1.4
    title:        daily mean fields from Global Ocean Physics Analysis and Fo...
    comment: 

### From A to C grid

In [8]:
ds_i = ds
_lat = ds.latitude
_lon = ds.longitude
_zt = ds.depth

ds_i = ds_i.rename({"depth": "k", "latitude":"j", "longitude":"i","so":"ssf", "thetao":"ttf"})
ds_i = ds_i.assign_coords(
    k=np.arange(ds_i.sizes["k"]),
    j=np.arange(ds_i.sizes["j"]),
    i=np.arange(ds_i.sizes["i"]),
    depth_t=("k", _zt.data),
    latitude_f = ("j", _lat.data),
    longitude_f = ("i", _lon.data),
)

## Calculate F and T mask
ds_i = ds_i.assign(fmask = ds_i.ssf.isel(time=0,drop=True).notnull())

ds_i = ds_i.assign(
    tmask=(
        ds_i.fmask.shift(i=0,j=0)
        | ds_i.fmask.shift(i=-1,j=-1).fillna(False)
        | ds_i.fmask.shift(i=0, j=-1).fillna(False)
        | ds_i.fmask.shift(i=-1,j=-1).fillna(False)
    ).astype(bool)
)

## PRIMARY: T and S at T points (cell centers) - this is the main placement
ds_i = ds_i.assign(
    tt_t = (ds_i.ttf.shift(i=-1,j=-1).fillna(0) + ds_i.ttf.shift(i=0,j=-1).fillna(0) + 
          ds_i.ttf.shift(i=-1,j=0).fillna(0) + ds_i.ttf.shift(i=0,j=0).fillna(0)) / 4,
    ss_t = (ds_i.ssf.shift(i=-1,j=-1).fillna(0) + ds_i.ssf.shift(i=0,j=-1).fillna(0) + 
          ds_i.ssf.shift(i=-1,j=0).fillna(0) + ds_i.ssf.shift(i=0,j=0).fillna(0)) / 4,
)

# ## OPTIONAL: Face values for advection (both tracers on both faces)
# ds_i = ds_i.assign(
#     # Temperature at U and V faces
#     ttu = (ds_i.tt.fillna(0) + ds_i.tt.shift(j=-1).fillna(0)) / 2,  # U face: avg in j
#     ttv = (ds_i.tt.fillna(0) + ds_i.tt.shift(i=-1).fillna(0)) / 2,  # V face: avg in i
    
#     # Salinity at U and V faces  
#     ssu = (ds_i.ss.fillna(0) + ds_i.ss.shift(j=-1).fillna(0)) / 2,  # U face: avg in j
#     ssv = (ds_i.ss.fillna(0) + ds_i.ss.shift(i=-1).fillna(0)) / 2,  # V face: avg in i
# )

# Rest of your code stays the same...
zt = ds_i.depth_t.data
zw = [zt[0]*2]

for k in range(1,50):
    zw.append((zt[k] - zw[k-1])*2 + zw[k-1])

ds_i = ds_i.assign_coords(depth_w = ("k",zw))

ds_i = ds_i.assign_coords(
    longitude_u = ds_i.longitude_f,
    latitude_v =  ds_i.latitude_f,
    latitude_u = ds_i.latitude_f + 1/12/2, 
    longitude_v = ds_i.longitude_f + 1/12/2,
    latitude_t = ds_i.latitude_f + 1/12/2, 
    longitude_t = ds_i.longitude_f + 1/12/2,
)

R = 6371e3 
ds_i = ds_i.assign_coords(
    dz_t = ds_i.depth_w - ds_i.depth_w.shift(k=1).fillna(0), 
    dx_t = np.deg2rad(1/12) * R * np.cos(np.deg2rad(ds_i.latitude_t)),
    dy_t = np.deg2rad(1/12) * R,
)

# Apply masks
ds_i['tt_t'] = ds_i.tt_t.where(ds_i.tmask)
ds_i['ss_t'] = ds_i.ss_t.where(ds_i.tmask)

# Clean up
ds_i = ds_i.drop_vars(['ttf','ssf','fmask','tmask'])
# ds_i

### create the invalid mask (land)

In [9]:
invalid_mask = ds_i.tt_t.isnull().all(dim=('k','time')).compute()

# temp_rolled = ds_i.tt_t.rolling(i=3, j=3, k=3, center=True, min_periods=1).mean()
# sal_rolled = ds_i.ss_t.rolling(i=3, j=3, k=3, center=True, min_periods=1).mean()

# temp_filled = xr.where(~invalid_mask, ds_i.tt_t, temp_rolled)
# sal_filled = xr.where(~invalid_mask, ds_i.ss_t, sal_rolled)

In [10]:
import os
import dask
from tqdm.dask import TqdmCallback

output_path = '/work/bk1450/b383184/Amazon/Atlantic/data/reanalysis/tracers'
os.makedirs(output_path, exist_ok=True)

def write_filled(varname_in, varname_out, fname):
    # Build the rolled mean lazily
    rolled = ds_i[varname_in].rolling(i=3, j=3, k=3, center=True, min_periods=1).mean()
    filled = xr.where(~invalid_mask, ds_i[varname_in], rolled).transpose('time','k','j','i')
    # Optional: downcast and rechunk for output
    filled = filled.astype('float32').chunk({'time': 1, 'k': 50, 'j': 201, 'i': 201})

    enc = {
        varname_out: {
            'zlib': True, 'shuffle': True, 'complevel': 1,
            'chunksizes': (1, 50, 201, 201),
        }
    }
    path = os.path.join(output_path, fname)
    task = filled.to_dataset(name=varname_out).to_netcdf(
        path, engine='h5netcdf', encoding=enc, compute=False
    )
    with TqdmCallback(desc=f"Writing {varname_out}"):
        dask.compute(task)

# Write temperature first, then salinity
write_filled('tt_t', 'tt_filled', f'T_{start_date[:7]}.nc')
write_filled('ss_t', 'ss_filled', f'S_{start_date[:7]}.nc')

Writing tt_filled:   0%|                                                                                                                                             | 0/24645 [00:00<?, ?it/s]

Writing tt_filled:   0%|▏                                                                                                                                 | 30/24645 [00:11<2:37:38,  2.60it/s]

Writing tt_filled:   1%|█▌                                                                                                                                 | 286/24645 [00:11<12:19, 32.93it/s]

Writing tt_filled:   1%|█▉                                                                                                                                 | 361/24645 [00:13<10:44, 37.67it/s]

Writing tt_filled:   2%|██                                                                                                                                 | 396/24645 [00:13<09:36, 42.07it/s]

...

Writing ss_filled:  99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎| 24486/24610 [07:49<00:06, 18.97it/s]

Writing ss_filled: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎| 24489/24610 [07:50<00:06, 18.07it/s]

Writing ss_filled: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 24492/24610 [07:50<00:06, 17.10it/s]

Writing ss_filled: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉| 24604/24610 [07:50<00:00, 217.85it/s]

Writing ss_filled: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24610/24610 [07:50<00:00, 52.31it/s]